In [ ]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q --upgrade transformers
!pip install -q pypdf
!pip install -q accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 8.8 MB/s eta 0:00:00


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
from google.colab import files

uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("Uploaded:", pdf_file)

Saving Module 56.pdf to Module 56.pdf
Uploaded: Module 56.pdf


In [ ]:
from pypdf import PdfReader

reader = PdfReader(pdf_file)

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text

print("Total characters:", len(text))

Total characters: 23253


In [ ]:
def chunk_text(text, chunk_size=500, overlap=100):

    words = text.split()

    chunks = []
    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total chunks:", len(chunks))

Total chunks: 9


In [ ]:
def clean_chunks(chunks):
    return list(set(chunks))


chunks = clean_chunks(chunks)
print("Unique chunks:", len(chunks))

Unique chunks: 9


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device=device
)

embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("Vectors stored:", index.ntotal)

Vectors stored: 1


In [ ]:
def retrieve(query, k=10):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        k
    )

    results = [chunks[i] for i in indices[0]]

    return results

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device=device
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def rerank(query, passages, top_k=3):

    pairs = [[query, p] for p in passages]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(passages, scores),
        key=lambda x: x[1],
        reverse=True
    )

    best_passages = [r[0] for r in ranked[:top_k]]

    return best_passages

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.2",
    device=0 if device == "cuda" else -1
)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
def build_prompt(question, context):

    prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question: {question}

Answer in one clear sentence.
If the answer is not in the context, say: Not found in document.
"""
    return prompt

In [ ]:
def answer_question(question):

    retrieved = retrieve(question)

    retrieved = clean_chunks(retrieved)

    best_chunks = rerank(question, retrieved)

    context = "\n".join(best_chunks[:2])  # limit context

    prompt = build_prompt(question, context)

    result = generator(
        prompt,
        max_new_tokens=100,
        do_sample=False
    )

    answer = result[0]["generated_text"]

    return answer, context

In [ ]:
question = "summary of this document"

answer, context = answer_question(question)

print("\nANSWER:\n")
print(answer)

print("\nUSED CONTEXT:\n")
print(context)